# 8교시. 실무 적용 시나리오 설계 및 최종 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/08_business_application.ipynb)

**이번 교시 행동:** 견적서·신청서·거래명세서 실물 사진을 비교하고 첫 PoC 한 가지를 고릅니다.

**통과 증거:** `course_outputs/poc_candidate_card.md`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/document_ai_lecture_2026/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded


In [ ]:
import io
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = lambda image: None

EXTENSION_IMAGE_PATHS = {
    "quotation": "sample_docs/extensions/quotation_photo.png",
    "application": "sample_docs/extensions/application_form_photo.png",
    "transaction_statement": (
        "sample_docs/extensions/transaction_statement_photo.png"
    ),
}
EXTENSION_EXAMPLES = {'quotation': {'name': '견적서', 'fields': ['문서번호', '공급자', '수신', '견적일', '품목', '총액'], 'rules': ['수량×단가=품목금액', '공급가액+부가세=총액'], 'risk': '총액 오류는 구매 의사결정에 직접 영향'}, 'application': {'name': '신청서', 'fields': ['신청번호', '신청자', '소속', '신청 과정', '승인'], 'rules': ['필수 동의', '관리자 승인 상태'], 'risk': '개인정보와 승인 누락을 사람이 확인'}, 'transaction_statement': {'name': '거래명세서', 'fields': ['문서번호', '공급자', '거래일', '품목', '세액', '총액'], 'rules': ['품목 합계=공급가액', '공급가액+세액=총액'], 'risk': '표 행·열 대응이 어긋나면 정산 오류'}}
extension_assets = load_course_assets(
    *EXTENSION_IMAGE_PATHS.values()
)
for key, path in EXTENSION_IMAGE_PATHS.items():
    image = Image.open(io.BytesIO(extension_assets[path])).convert("RGB")
    image.thumbnail((320, 400))
    print(key, image.size)
    display(image)


## 형식이 바뀌면 생기는 어려움

- **Excel**: 수식, 병합 셀, 숨김 시트, 숫자 서식
- **Word**: 머리글, 텍스트박스, 변경 추적, 이미지로 삽입된 본문
- **PDF**: 텍스트·스캔 혼합 페이지, 암호, 깨진 문자맵
- **PPT**: 그룹 도형, 읽기 순서, 발표자 노트
- **표 캡처**: 셀 관계가 사라져 행·열 위상을 다시 복원해야 함


In [ ]:
import re
import zipfile

OFFICE_PATHS = [
    "sample_docs/formats/quotation.xlsx",
    "sample_docs/formats/application_form.docx",
    "sample_docs/formats/transaction_statement.pdf",
    "sample_docs/formats/table_summary.pptx",
]
office_assets = load_course_assets(*OFFICE_PATHS)
office_dir = OUTPUT_DIR / "office_format_samples"
office_dir.mkdir(exist_ok=True)
for path, payload in office_assets.items():
    (office_dir / Path(path).name).write_bytes(payload)

def xml_text_count(path, prefix, text_tag):
    with zipfile.ZipFile(path) as archive:
        names = [
            name for name in archive.namelist()
            if name.startswith(prefix) and name.endswith(".xml")
        ]
        text_count = 0
        for name in names:
            xml = archive.read(name).decode("utf-8", errors="ignore")
            text_count += len(re.findall(text_tag, xml))
        return len(names), text_count

xlsx_sheets, xlsx_values = xml_text_count(
    office_dir / "quotation.xlsx",
    "xl/worksheets/",
    r"<x:(?:v|f)>",
)
docx_parts, docx_text = xml_text_count(
    office_dir / "application_form.docx",
    "word/document",
    r"<w:t",
)
pptx_slides, pptx_text = xml_text_count(
    office_dir / "table_summary.pptx",
    "ppt/slides/slide",
    r"<a:t>",
)
pdf_bytes = (office_dir / "transaction_statement.pdf").read_bytes()
print("Excel:", xlsx_sheets, "개 시트 XML · 값/수식", xlsx_values)
print("Word:", docx_text, "개 본문 텍스트 run · 이미지 본문 여부 확인")
print("PDF:", pdf_bytes[:5], "· 텍스트층 샘플")
print("PPT:", pptx_slides, "개 슬라이드 · 텍스트", pptx_text)

office_bundle = OUTPUT_DIR / "office_format_samples.zip"
with zipfile.ZipFile(office_bundle, "w") as archive:
    for path in sorted(office_dir.iterdir()):
        archive.write(path, path.name)
print("실제 파일 4종 묶음:", office_bundle)
download_artifact(office_bundle)


## 내가 직접 만드는 PoC 카드

`candidate`는 `quotation`, `application`, `transaction_statement`
중 하나입니다. 점수는 1~5점이며 오류 영향과 예외 빈도는 낮을수록
첫 PoC에 유리합니다.


In [ ]:
# TODO: 내 업무 후보와 점수·검토자·중단 조건을 채우세요.
candidate = None
score = {
    "반복량": None,
    "필드 안정성": None,
    "오류 영향": None,
    "예외 빈도": None,
    "사람 검토 가능성": None,
}
review_owner = None
stop_condition = None
if candidate is None or any(value is None for value in score.values()):
    print("빈칸이 있습니다. 아래 힌트·전체 정답과 비교하세요.")


<details>
<summary>힌트와 전체 정답 보기</summary>

예시는 거래명세서를 한 장씩 처리하고 정산 담당자가 검토하는 작은
PoC입니다. 값이 맞지 않거나 원본 근거가 없으면 저장을 중단합니다.
</details>


In [ ]:
from textwrap import dedent

candidate = candidate or "transaction_statement"
if candidate not in EXTENSION_EXAMPLES:
    raise ValueError(
        "candidate는 quotation, application, "
        "transaction_statement 중 하나여야 합니다."
    )
default_score = {
    "반복량": 4,
    "필드 안정성": 4,
    "오류 영향": 2,
    "예외 빈도": 3,
    "사람 검토 가능성": 5,
}
score = {
    key: (
        int(value)
        if value is not None
        else default_score[key]
    )
    for key, value in score.items()
}
if not all(1 <= value <= 5 for value in score.values()):
    raise ValueError("모든 점수는 1~5 사이여야 합니다.")
review_owner = review_owner or "정산 담당자"
stop_condition = (
    stop_condition
    or "필수값·합계·원본 근거 중 하나라도 틀리면 자동 저장 중단"
)
example = EXTENSION_EXAMPLES[candidate]
recommendation = (
    "GO_SMALL"
    if (
        score["반복량"] >= 4
        and score["필드 안정성"] >= 3
        and score["오류 영향"] <= 3
        and score["예외 빈도"] <= 3
        and score["사람 검토 가능성"] >= 4
    )
    else "REVIEW"
)
card = f'''# 문서 자동화 PoC 후보 카드

| 항목 | 내용 |
| --- | --- |
| 선택 문서 | {example["name"]} |
| 추출 필드 | {", ".join(example["fields"])} |
| 검증 규칙 | {" / ".join(example["rules"])} |
| 틀렸을 때 영향 | {example["risk"]} |
| 입력 제한 | 승인된 비식별 한 장 |
| 최종 산출물 | 사람 승인 후 Excel |
| 사람 검토자 | {review_owner} |
| 중단 조건 | {stop_condition} |
| 점수 | {" / ".join(f"{key} {value}" for key, value in score.items())} |
| 제안 | {recommendation} |

## 첫 PoC 통과 기준

- 같은 양식 30장을 모아 정답표와 비교한다.
- 필드별 정확도뿐 아니라 수정률과 처리시간을 기록한다.
- 오류 시 자동 저장하지 않고 검토 대기열로 보낸다.
- 개인정보·보존·삭제 정책을 먼저 승인받는다.
'''
output_path = OUTPUT_DIR / "poc_candidate_card.md"
output_path.write_text(dedent(card), encoding="utf-8")
print(dedent(card))
print("CHECKPOINT 1/1 PASS:", output_path)
download_artifact(output_path)
